# TraceGen — compare your Layer-2 run with the paper

`L2_eval.sh` runs the six released checkpoints on the full Droid and EpicKitchen test splits, and
`L2_eval_baseline.sh` runs the TraceGen authors' released checkpoints with the authors' own code.
This notebook recomputes the table from those outputs with the **same metric code as Layer 1**
(`tg_l1.py`) and prints paper vs Layer 1 (the paper's predictions) vs Layer 2 (your predictions).

Rows whose run has not finished yet are shown as `pending`.

The notebook reads only small per-sample files (`L2_outputs/<run>/per_sample_errors.csv` and `L2_outputs/L1_cache/`),
so it runs in about a minute. Write them once with `python apps/tracegen/L2_cache.py` after your runs finish
(it reads each run's 25-34 GB of sample files once).

In [ ]:
import json

import numpy as np
import pandas as pd

import tg_l1
import tg_l2

print("L2 outputs:", tg_l2.l2_root())
for (row, env), name in tg_l2.L2_RUNS.items():
    run = tg_l2.finished_run(name)
    print(f"{row:22s} {env:5s} {run.name if run else 'pending'}")

In [ ]:
rows, per_sample_l2 = [], {}
for (row, env), name in tg_l2.L2_RUNS.items():
    l1_df = tg_l2.l1_per_sample(row, env)
    l1 = tg_l1.aggregate(l1_df)
    l1_orig = json.load(open(tg_l1.l1_root() / "runs" / tg_l1.RUNS[(row, env)] / "metrics.json"))
    assert len(l1_df) == tg_l1.N_TOTAL[env] and abs(l1["MAE"] - l1_orig["trajectory_mae_x100"]) < 1e-5, (row, env)
    run = tg_l2.finished_run(name)
    l2 = None
    if run is not None:
        df = tg_l2.run_per_sample(run)
        per_sample_l2[(row, env)] = df
        l2 = tg_l1.aggregate(df)
        assert l2["n_valid"] == tg_l1.N_KEPT[env], (row, env, l2["n_valid"])
        orig = json.load(open(run / "metrics.json"))  # written by the evaluation script itself
        for metric, key in (("MSE", "trajectory_mse_x100"), ("MAE", "trajectory_mae_x100"), ("Endp.", "endpoint_mse_x100")):
            assert abs(l2[metric] - orig[key]) < 1e-5, (row, env, metric, l2[metric], orig[key])
    for metric in ("MSE", "MAE", "Endp."):
        col = ("Dr " if env == "droid" else "Ep ") + metric
        paper = tg_l1.PAPER[row][tg_l1.COLUMNS.index(col)]
        rows.append(dict(row=row, column=col, paper=paper, L1=l1[metric],
                         L2=l2[metric] if l2 else np.nan,
                         **{"L2-L1 (%)": 100 * (l2[metric] - l1[metric]) / l1[metric] if l2 else np.nan}))
cmp = pd.DataFrame(rows)
pd.set_option("display.width", 200)
print(cmp.to_string(index=False, float_format=lambda v: f"{v:.4f}", na_rep="pending"))

## Recomputed table from fresh outputs only (x100)

In [ ]:
wide = cmp.pivot(index="row", columns="column", values="L2").reindex(
    index=list(dict.fromkeys(r for r, _ in tg_l2.L2_RUNS)), columns=tg_l1.COLUMNS)
print(wide.to_string(float_format=lambda v: f"{v:.3f}", na_rep="pending"))

## Baseline: TraceGen authors' checkpoint

The paper quotes the TraceGen paper's own numbers. `L2_eval_baseline.sh` evaluates the authors' released
checkpoints with the authors' evaluation code.

In [ ]:
rows = []
for env in tg_l1.ENVS:
    run = tg_l2.baseline_metrics(env)
    for metric in ("MSE", "MAE", "Endp."):
        col = ("Dr " if env == "droid" else "Ep ") + metric
        rows.append(dict(column=col, paper_quoted=tg_l1.PAPER["TraceGen (quoted)"][tg_l1.COLUMNS.index(col)],
                         L2_authors_code=run[metric] if run else np.nan))
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4f}", na_rep="pending"))

## Per-sample agreement between Layer 1 and Layer 2

Each point is one test sample: its MAE with the original predictions (x) and with the fresh predictions (y).

In [ ]:
import matplotlib.pyplot as plt

done = list(per_sample_l2)
if not done:
    print("no finished Layer-2 run yet")
else:
    fig, axes = plt.subplots(1, len(done), figsize=(4 * len(done), 3.6), squeeze=False)
    for ax, (row, env) in zip(axes[0], done):
        a = tg_l2.l1_per_sample(row, env).query("kept")
        b = per_sample_l2[(row, env)].query("kept")
        x = 100 * a["sae"].to_numpy() / a["n_coord"].to_numpy()
        y = 100 * b["sae"].to_numpy() / b["n_coord"].to_numpy()
        ax.scatter(x, y, s=2, alpha=0.3)
        lim = np.percentile(np.r_[x, y], 99.5)
        ax.plot([0, lim], [0, lim], "k--", lw=0.8)
        ax.set_xlim(0, lim); ax.set_ylim(0, lim)
        ax.set_title(f"{row}, {env}"); ax.set_xlabel("L1 per-sample MAE x100"); ax.set_ylabel("L2")
    plt.tight_layout()
    plt.show()